# Pre-release model predicting the IMDB rating and number of IMDB votes

***Disclaimer: This analysis is based on data from 21.03.2026. Ratings, vote counts, etc. may have changed since then and new titles might have been added to the data. As a result, some descriptive text, e.g. about the numbers of missing values, might not be accurate when using newer data.***

## Table of contents

1.  [Introduction](#1-Introduction)
2.  [Data loading & EDA](#2-Data-loading-&-EDA)
3.  [Feature Engineering](#3-Feature-Engineering)
4.  [Random Forest Training & Results](#4-Random-Forest-Training-&-Results)
5.  [XGBoost Training & Results](#5-XGBoost-Training-&-Results)
6.  [Comparison & Conclusions](#6-Comparison-&-Conclusions)

## 1. Introduction
With the IMDB (and TMDB) data at hand, we want to build a model to predict some titles IMDB rating and the number of its IMDB votes. We will start with a RandomForest regression model and also try an XGBoost model.

Since we view it as a predictive model that should not make use of post-release data, we do not make use of the revenue and obviously, we do not make use of the TMDB rating (and TMDB votes) either, because these are too predictive and are not accessible pre-release either.

In addition, since the goal of this project is not to make a very strong predictive model, we only focus on movies with at least 10.000 IMDB votes.

In [3]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 2. Data loading & EDA

### Data loading

First, we load the data from the 'excerpt_data.parquet' file located in '/processed_data/'. This file is not uploaded to the git repository, but it can be created by executing the general data pipeline by executing the '/src/prepare_all_data.py' python script. However, this takes a few hours to execute due to the large number of TMDB API calls to be done.

In [4]:
df = pd.read_parquet('processed_data/excerpt_data.parquet')

Let us filter the predictor and the target variables. The columns we drop are either non-predictive (like tconst, or the IMDB link), is post-release information (like the revenue or the TMDB rating/votes). While the movie title might be predictive to some degree (description texts might be more predictive), analyzing it goes beyond the scope of this notebook.

Also, we quickly already rescale the number of IMDB votes to a log-scale, even though it is rather a part of rescaling later.

In [5]:
X = df.drop(columns=['tconst', 'Link', 'PrimaryTitle', 'OriginalTitle', 'Type', 'EndYear', 'IMDBRating', 'IMDBVotes', 'TMDBRating', 'TMDBVotes', 'Revenue'])
y = df.loc[:, ['IMDBRating', 'IMDBVotes']]
y['IMDBVotes'] = np.log(y.loc[:, 'IMDBVotes'])

### Exploratory data analysis (EDA)

Let us see what the training data looks like now. Below, we see that many columns contain lists (or rather numpy.ndarrays) of strings that we will encode later on.

In [6]:
X.head()

,OriginalLanguage,OriginCountry,StartYear,Genres,Runtime,Budget,Adult,Directors,Writers,Actors,Actresses,Cinematographers,Composers
0,en,[US],2008,[Horror],101,0.0,0,[Marcel Sarmiento],[Trent Haaga],"[Shiloh Fernandez, Noah Segan, Eric Podnar, An...","[Candice King, Jenny Spain, Susan Marie Keller]",[Harris Charalambous],[Joseph Bauer]
1,en,[US],2007,"[Crime, Thriller]",88,25000000.0,0,[Renny Harlin],[Matthew Aldrich],"[Samuel L. Jackson, Ed Harris, Luis Guzmán, Jo...","[Eva Mendes, Keke Palmer, Maggie Lawson]",[Scott Kevan],"[Cory Chisel, Richard Gibbs]"
2,en,"[CA, US]",2010,"[Biography, Crime, Drama]",112,0.0,0,[Larysa Kondracki],"[Larysa Kondracki, Eilis Kirwan]","[David Strathairn, Nikolaj Lie Kaas, Alexandru...","[Rachel Weisz, Monica Bellucci, Vanessa Redgra...",[Kieran McGuigan],[Mychael Danna]
3,en,[US],2007,"[Horror, Mystery, Thriller]",105,12000000.0,0,[Chris Sivertson],[Jeff Hammond],"[Neal McDonough, Michael Adler, Michael Esparz...","[Lindsay Lohan, Lindsay Lohan, Julia Ormond, B...",[John R. Leonetti],[Joel McNeely]
4,en,[US],2020,"[Comedy, Horror, Mystery]",103,14000000.0,0,[Steven Brill],"[Tim Herlihy, Adam Sandler]","[Adam Sandler, Kevin James, Ray Liotta, Steve ...","[Julie Bowen, Maya Rudolph, June Squibb]",[Seamus Tierney],[Rupert Gregson-Williams]


In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12450 entries, 0 to 12449
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   OriginalLanguage  12446 non-null  object 
 1   OriginCountry     12450 non-null  object 
 2   StartYear         12450 non-null  Int64  
 3   Genres            12450 non-null  object 
 4   Runtime           12450 non-null  Int64  
 5   Budget            12445 non-null  Float64
 6   Adult             12450 non-null  Int64  
 7   Directors         12450 non-null  object 
 8   Writers           12450 non-null  object 
 9   Actors            12450 non-null  object 
 10  Actresses         12450 non-null  object 
 11  Cinematographers  12450 non-null  object 
 12  Composers         12450 non-null  object 
dtypes: Float64(1), Int64(3), object(9)
memory usage: 1.3+ MB


Let us quickly declare lists of the different columns that require different care in future processes.

In [8]:
numerical_features = X.select_dtypes(include=['Int64', 'Float64']).columns
categorical_features = ['OriginalLanguage']
list_features = X.select_dtypes(include=['object']).columns.drop('OriginalLanguage')

In the X.info() call above, we can see that the 'OriginalLanguage' column has 4 missing values and the 'Budget' column has 5 missing values. In fact, the 'Budget' column has much more missing values (3986 more, see below). The TMDB API sometimes does return a budget of 0, but, obviously, this is missing data as well. No movie costs nothing.

In [9]:
print(X['Budget'].apply(lambda x: x == 0).value_counts())

Budget
False    8464
True     3986
Name: count, dtype: int64


Furthermore, the columns containing lists do have missing data of empty lists (which are not declared as NA type by pandas). Since we constructed the dataset, we know this precisely. Below, we see how empty lists there are in each column. We will adress these missing values in the next section.

In [10]:
for col in list_features:
    print(f'{col}: {X[col].apply(lambda x: len(x)==0).sum()} empty lists')

OriginCountry: 25 empty lists
Genres: 0 empty lists
Directors: 9 empty lists
Writers: 135 empty lists
Actors: 222 empty lists
Actresses: 549 empty lists
Cinematographers: 377 empty lists
Composers: 743 empty lists


## 3. Feature Engineering

### Missing Values

The 'OriginalLanguage' column contains 4 missing values of Pythons None type. We simply replace these by a string 'Unknown'.

In [11]:
types = X['OriginalLanguage'].apply(lambda x: type(x)).value_counts()
print(types)

X['OriginalLanguage'] = X['OriginalLanguage'].fillna('Unknown')

OriginalLanguage
<class 'str'>         12446
<class 'NoneType'>        4
Name: count, dtype: int64


The numerical features mostly contain no missing values. Only the 'Budget' column contains 5 missing values but, more importantly, contains budgets of 0 given out by the TMDB data. These obviously represent missing values as well and thus, we create a missing value flag column 'HasBudget' for the budget and set all the missing values to the median. In addition, we add a column 'Budget_log' that shows the logarithm of the budget as it is highly skewed so that the models work better.

In [12]:
print(X[numerical_features].isna().sum())

print('\n Number of 0 values in the "Budget" column: ' + str(X['Budget'].apply(lambda x: x==0).sum()))

X['Budget'] = X['Budget'].fillna(0)
X['HasBudget'] = (X['Budget'] != 0).astype('Int64')

median = X['Budget'].median()
X['Budget'] = X['Budget'].replace(0, median)

X['Budget_log'] = np.log(X['Budget'])

StartYear    0
Runtime      0
Budget       5
Adult        0
dtype: int64

 Number of 0 values in the "Budget" column: 3986


The columns containing lists should be split up. The 'OriginCountry' (99 unique values) and the 'Genres' (25 unique values) column are fine for multi-hot encoding later (see numbers below), but the columns of people involved in the movie production contain way too many people (33843 unique actors).

In [13]:
multi_label_features = ['Genres', 'OriginCountry']
people_features = ['Directors', 'Actors', 'Actresses', 'Writers', 'Composers']

In [14]:
print('The "Genres" column has ' + str(len(X['Genres'].explode().unique())) + ' unique genres.')
print('The "OriginCountry" column has ' + str(len(X['OriginCountry'].explode().unique())) + ' unique countries.')
print()
print('The "Actors" column has ' + str(len(X['Actors'].explode().unique())) + ' unique actors. It is the largest of the people features.')

for col in multi_label_features:
    X[col] = X[col].apply(lambda x: [col + '_Unknown'] if len(x)==0 else x)

The "Genres" column has 25 unique genres.
The "OriginCountry" column has 99 unique countries.

The "Actors" column has 33843 unique actors. It is the largest of the people features.


The other columns containing lists have quite some empty lists as seen below.

In [15]:
for col in people_features:
    print(col, X[col].apply(lambda x: len(x)==0).sum())

Directors 9
Actors 222
Actresses 549
Writers 135
Composers 743


In all cases of lists, we replace missing values, i.e. empty lists, by e.g. ['Genres_Unknown']. For the columns of people involved in the movie production, we also create missing value flag columns for each.

In [16]:
for col in people_features:
    X['Has' + col] = X[col].apply(lambda x: len(x)!=0).astype('Int64')
    X[col] = X[col].apply(lambda x: np.array([col + '_Unknown']) if len(x)==0 else x)

### Datatypes

Most columns are fine for later use. However, the peoples columns containing lists have to be lists. In the original dataset, every entry was a numpy array. Above, we changed the empty ones to lists. Now, we convert all of them to lists.

In [17]:
for col in people_features:
    print(col, X[col].apply(lambda x: type(x)).value_counts())

Directors Directors
<class 'numpy.ndarray'>    12450
Name: count, dtype: int64
Actors Actors
<class 'numpy.ndarray'>    12450
Name: count, dtype: int64
Actresses Actresses
<class 'numpy.ndarray'>    12450
Name: count, dtype: int64
Writers Writers
<class 'numpy.ndarray'>    12450
Name: count, dtype: int64
Composers Composers
<class 'numpy.ndarray'>    12450
Name: count, dtype: int64


In [18]:
def ensure_list(x):
    if isinstance(x, list):
        return x
    elif isinstance(x, np.ndarray):
        return x.tolist()
    
# for col in list_features:
#     X[col] = X[col].apply(ensure_list)

### Rescaling and Encoding

We want to make use of scikit-learns ColumnTransformer class inside the pipeline. For the numerical columns (that will be rescaled) and the One-Hot encoded 'OriginalLanguage' column, we can use the predefined StandardScaler and OneHotEncoder classes from scikit-learn, respectively. However, both the multi-hot encoded columns of 'Genres' and 'OriginCountry' and the people columns require custom encoders.

First, we build a custom wrapper around the MultiLabelBinarizer class from scikit-learn for 'Genres' and 'OriginCountry'.

In [19]:
class MultiHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        # X may be a DataFrame with one column causing RandomForest to crash, so we need to flatten it
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]
        self.mlb.fit(X)
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{input_features[0]}__{cls}" for cls in self.mlb.classes_])


Next, for the people columns which have high cardinality, we build a custom aggregator function. Each movie title has a number of e.g. actors involved in the production. For each actor, we build the mean, count, min and max of IMDB ratings (and votes) of titles they have worked on. Then, for some given movie title, we build the average over all of these actors ending up with new (numerical) columns named e.g. 'Actors_IMDBRating_mean'.

In [20]:
class PeopleAggregator(BaseEstimator, TransformerMixin):
    def __init__(self, people_cols, stats=('mean', 'count', 'min', 'max')):
        self.people_cols = people_cols
        self.stats = stats

    def fit(self, X, y):
        self.target_names_ = y.columns
        self.agg_ = {}
        self.feature_names_ = []

        for col in self.people_cols:
            exploded = X[[col]].explode(col)
            exploded = exploded.join(y)

            # Compute stats
            agg = exploded.groupby(col)[self.target_names_].agg(self.stats)

            # Flatten MultiIndex columns
            agg.columns = [
                f"{col}_{target}_{stat}"
                for target, stat in agg.columns
            ]

            self.agg_[col] = agg

            # Store feature names for this column
            self.feature_names_.extend(agg.columns)

        return self

    def transform(self, X):
        outputs = []

        for col in self.people_cols:
            col_stats = self.agg_[col]
            global_means = col_stats.mean()

            movie_features = []
            for lst in X[col]:
                person_stats = []

                for p in lst:
                    if p in col_stats.index:
                        person_stats.append(col_stats.loc[p].values)
                    else:
                        person_stats.append(global_means.values)

                movie_features.append(np.mean(person_stats, axis=0))

            outputs.append(np.vstack(movie_features))

        return np.hstack(outputs)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_)

Since we introduced many new numerical columns (e.g. the 'Has...' flag columns or the log-transformed Budget), we need to update the numerical_features list.

In [21]:
numerical_features = X.select_dtypes(include=['Int64', 'Float64']).columns

Finally, we put everything together in a ColumnTransformer that handles each column in the way we implemented.

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('genres', MultiHotEncoder(), ['Genres']),
        ('origincountry', MultiHotEncoder(), ['OriginCountry']),
        ('people', PeopleAggregator(people_features), people_features)
    ],
    remainder='drop'
)

## 4. Random Forest Training & Results

### Building the pipeline

Now that we have the ColumnTransformer set up as preprocessor, we can build the full pipeline with the RandomForestRegressor

In [23]:

model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('regressor', RandomForestRegressor())
])

### Train/Test split

At this point, we need the train/test split, of course. We will use this later on for the XGBoost as well.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Now that everything is set up, we can get to model training and evaluation. To start, we train the model once and see if everything works.

### Single model run

In [25]:
model.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


After fitting the model on the training data, we use it to predict the IMDB rating (and votes) on the testing data. Note, that we quickly transform the logarithmic IMDB votes (see creation of y at the very beginning) back to the normal scale.

In [26]:
y_pred = model.predict(X_test)
y_pred[:, 1] = np.exp(y_pred[:, 1])

/home/henning/.venvs/dev/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['AW', 'CY'] will be ignored
  warnings.warn(


#### Evaluation

For this, we investigate the root mean squared error (RMSE) and the mean absolute error (MAE) for both the IMDB rating and the IMDB votes. Here, we write a custom function that we can use for later evaluation again.

In [27]:
def evaluate(y_test, y_pred):
    rmse_rating = np.sqrt(mean_squared_error(y_test['IMDBRating'], y_pred[:, 0]))
    rmse_votes  = np.sqrt(mean_squared_error(y_test['IMDBVotes'],  y_pred[:, 1]))

    mae_rating = mean_absolute_error(y_test['IMDBRating'], y_pred[:, 0])
    mae_votes  = mean_absolute_error(y_test['IMDBVotes'],  y_pred[:, 1])

    print("RMSE Rating:", rmse_rating)
    print("RMSE Votes:", rmse_votes)
    print("MAE Rating:", mae_rating)
    print("MAE Votes:", mae_votes)

    return rmse_rating, rmse_votes, mae_rating, mae_votes

In [28]:
rfs_rmse_rating, rfs_rmse_votes, rfs_mae_rating, rfs_mae_votes = evaluate(y_test, y_pred)

RMSE Rating: 0.8852981339321658
RMSE Votes: 71253.56039030764
MAE Rating: 0.6669240963855421
MAE Votes: 47348.94023247776


Let us see what features are the most important ones. Below, we print the features with their relative relevance in descending order.We can clearly see that the averaged IMDB rating (and votes) of the actors are by far the most important features with 29.45% and 36.43%, respectively.

In [29]:
rf = model.named_steps['regressor']
feature_names = model.named_steps['preprocess'].get_feature_names_out()

importances = sorted(
    zip(feature_names, rf.feature_importances_),
    key=lambda x: x[1],
    reverse=True
)

for name, score in importances[:10]:
    print(f"{name}: {score:.4f}")

people__Actors_IMDBVotes_mean: 0.3637
people__Actors_IMDBRating_mean: 0.2924
people__Writers_IMDBVotes_max: 0.0424
people__Writers_IMDBRating_mean: 0.0379
people__Actors_IMDBRating_min: 0.0367
people__Writers_IMDBVotes_mean: 0.0288
people__Actors_IMDBVotes_min: 0.0179
people__Writers_IMDBVotes_min: 0.0178
people__Actors_IMDBVotes_max: 0.0161
people__Actresses_IMDBVotes_max: 0.0136


### Grid search cross-validation

First, we define the grid of hyperparameters that we want to search.

*n_estimators* : number of trees in the forest  
*max_depth* : maximum number of splits each tree performs  
*min_samples_split* : minimum number of samples (titles) to allow for a split  
*min_samples_leaf* : minimum number of samples (titles) that make up a leaf (last set of titles)  
*max_features* : maximum number of features considered at each split

In [30]:
param_grid = {
    'regressor__n_estimators': [300, 600],
    'regressor__max_depth': [None, 30],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2],
    'regressor__max_features': ['sqrt',  None]
}

And set up the GridSearchCV class using both the pipeline and the parameter grid.

In [31]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

#### Model training & evaluation

*Note: The grid search prints out annoying errors like 'UserWarning: unknown class(es) ['AS', 'BA', 'BH', 'BM', 'BO', 'BW'] will be ignored' which we suppress by the following lines of code. It simply means that during cross-validation, there are labels in the validation fold that were not present in the training data. Since there are quite some countries in the list with very few titles (even down to 1), this happens a lot.*

In [32]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

We can now train the model with the grid search and the cross-validation...

In [33]:
grid.fit(X_train, y_train)

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'regressor__max_depth': [None, 30], 'regressor__max_features': ['sqrt', None], 'regressor__min_samples_leaf': [1, 2], 'regressor__min_samples_split': [2, 5, ...], ...}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


and again, fit the testing data.

In [34]:
y_pred = grid.predict(X_test)
y_pred[:,1] = np.exp(y_pred[:,1])

Again, we evaluate the results and see that they improved, but only slightly. So, we will also try another algorithm, the extreme gradient boost (XGBoost).

In [35]:
rfg_rmse_rating, rfg_rmse_votes, rfg_mae_rating, rfg_mae_votes = evaluate(y_test, y_pred)

RMSE Rating: 0.8589688024238898
RMSE Votes: 68253.36640243045
MAE Rating: 0.6414473101815704
MAE Votes: 47553.577828516485


## 5. XGBoost Training & Results

Since the XGBoost algorithm is not implemented in scikit-learn, we need to import it from another module. However, it is comletely compatible with the structure of scikit-learns algorithms. In particular, it can be used in the Pipeline class and with that, we can perform a GridSearchCV as well.

In [36]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

In [37]:
regressor = MultiOutputRegressor(
        XGBRegressor(
            objective='reg:squarederror',
            tree_method='hist',
            learning_rate=0.05,
            n_estimators=500,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            reg_alpha=0.0,
            n_jobs=-1
        )
    )

model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('regressor', regressor)
])

### Single model run

As before, we train the model once to test if everything works and to obtain a baseline working performance.

In [38]:
model.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [39]:
y_pred = model.predict(X_test)
y_pred[:,1] = np.exp(y_pred[:,1])

Evaluation with the RMSE and the MAE shows that it actually performs worse than the single run of random forest. Of course, this can simply be an unfortunate set of hyperparameters, so we continue with a grid search cross-validation run as well.

In [40]:
xgs_rmse_rating, xgs_rmse_votes, xgs_mae_rating, xgs_mae_votes = evaluate(y_test, y_pred)

RMSE Rating: 0.8941549220404007
RMSE Votes: 73342.48241828277
MAE Rating: 0.6769754561841727
MAE Votes: 48261.707013819854


### Grid search cross-validation

Instead of training the trees in parallel, like the random forest algorithm does, the XGBoost trains trees sequentially to correct potential mistakes from the trees before. So, again, it constructs a family of decision trees TODO. We choose the following parameter grid with parameters

*n_estimators* : number of trees to build  
*max_depth* : maximum depth of each tree  
*learning_rate* : factor to rescale the influence of the new tree relative to the previous one  
*subsample* : fraction of samples per tree  
*colsample_bytree* : fraction of features used for each tree

In [41]:
param_grid = {
    'regressor__estimator__n_estimators': [300, 600],
    'regressor__estimator__max_depth': [4, 6, 8],
    'regressor__estimator__learning_rate': [0.03, 0.05, 0.1],
    'regressor__estimator__subsample': [0.7, 0.9],
    'regressor__estimator__colsample_bytree': [0.7, 0.9],
}


In [42]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

#### Model training & evaluation

Again, we train the model on the training data over the whole grid...

In [43]:
grid.fit(X_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


,estimator,"Pipeline(step...None, ...)))])"
,param_grid,"{'regressor__estimator__colsample_bytree': [0.7, 0.9], 'regressor__estimator__learning_rate': [0.03, 0.05, ...], 'regressor__estimator__max_depth': [4, 6, ...], 'regressor__estimator__n_estimators': [300, 600], ...}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


and predict the testing data...

In [44]:
y_pred = grid.predict(X_test)
y_pred[:,1] = np.exp(y_pred[:,1])

before we evaluate the RSME and the MAE for the XGBoost model on the grid search.

In [45]:
xgg_rmse_rating, xgg_rmse_votes, xgg_mae_rating, xgg_mae_votes = evaluate(y_test, y_pred)

RMSE Rating: 0.8752940809093859
RMSE Votes: 72739.65999355078
MAE Rating: 0.6598436030422348
MAE Votes: 48094.69724388072


## 6. Comparison & Conclusions

Below, we compare the metrics for the different approaches.

In [46]:
eval_array = np.array([
    [rfs_rmse_rating, rfs_mae_rating, rfs_rmse_votes, rfs_mae_votes],
    [rfg_rmse_rating, rfg_mae_rating, rfg_rmse_votes, rfg_mae_votes],
    [xgs_rmse_rating, xgs_mae_rating, xgs_rmse_votes, xgs_mae_votes],
    [xgg_rmse_rating, xgg_mae_rating, xgg_rmse_votes, xgg_mae_votes],
])

eval_df = pd.DataFrame(
    eval_array,
    columns=['RMSE Rating', 'MAE Rating', 'RMSE Votes', 'MAE Votes'],
    index=['Random Forest (Single run)', 'Random Forest (Grid Search)', 'XGBoost (Single run)', 'XGBoost (Grid Search)']
)

eval_df

,RMSE Rating,MAE Rating,RMSE Votes,MAE Votes
Random Forest (Single run),0.885298,0.666924,71253.560390,47348.940232
Random Forest (Grid Search),0.858969,0.641447,68253.366402,47553.577829
XGBoost (Single run),0.894155,0.676975,73342.482418,48261.707014
XGBoost (Grid Search),0.875294,0.659844,72739.659994,48094.697244


We see that the grid searches (with cross-validation) improved both models as is expected. In addition, we see that the random forest performs slightly better in both the IMDB rating and the votes, but the difference is pretty small.

The best model is the random forest with tuned hyperparameters and produces an MAE of 0.64 for the IMDB rating and approximately 47.400 IMDB votes.